# Agentic RAG

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
load_dotenv()

True

## Retrieval-Augmented Generation
LLM models are trained on large amounts of publicly available datasets. The information found in these datasets constitutes the intrinsic knowledge that a model can reason over and answer questions about. To generate business value, organizations need AI models to interact with proprietary data which was not available during model training.

RAG is a method for giving an LLM access to an external knowledge base that extends to model's capabilities to fit a specific purpose. In a RAG flow, the user query is first used to search for potential matches in the external database (retrieval). The matching documents form the context which the LLM will use to provide more specific and accurate answers than it would without the additional knowledge (augmentation).

In [3]:
def get_response(client, prompt):
	response = client.chat.completions.create(
		model="gpt-4o-mini",
		messages=[{"role": "user", "content": prompt}],
		user="llm-zoomcamp",
		stream=False
	)

	return response.choices[0].message.content

In [4]:
openai_client = OpenAI()

In [5]:
get_response(openai_client, "Who won yesterday's NBA finals game?")

"I'm sorry, but I don't have access to real-time data or updates. To find out the latest results from the NBA finals game, please check a reliable sports news website or the NBA's official website."

## Dataset

In [11]:
import requests

In [12]:
docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [16]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
	course_url = f'{url_prefix}/{course["path"]}'

	course_response = requests.get(course_url)
	course_response.raise_for_status()
	course_data = course_response.json()

	documents.extend(course_data)

In [17]:
len(documents)

1208

In [18]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

## Search
Context scope limitation
- lowers LLM costs by reducing the amount of tokens passed to the model
- minimizes the probability of irrelevant model outputs
- enables faster model responses

Popular search engines:
- Lucine
- ElsticSearch
- Apache Solr
- minsearch

In [19]:
from minsearch import Index

In [20]:
index  = Index(
	text_fields=['question', 'section', 'answer'],
	keyword_fields=['course']
)

In [21]:
index.fit(documents)

In [26]:
def search(question):
	return index.search(
		question,
		filter_dict={'course': 'llm-zoomcamp'},
		num_results=5,
		boost_dict={'question': 2.0, 'section': .5}
	)

In [27]:
question = "i just discovered the course, can I still join?"
search(question)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learn